# Vehicle Miles Travelled Prediction using RNN (LSTM)
This notebook predicts miles travelled using time-series data.

In [ ]:
!pip install numpy pandas matplotlib scikit-learn tensorflow

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

## Load Dataset
Replace this with your dataset CSV file.
Expected format: Date, Miles

In [ ]:
# Example synthetic dataset (if you don't have one)
dates = pd.date_range(start='2020-01-01', periods=500)
miles = np.cumsum(np.random.normal(5, 2, 500))  # simulated travel

data = pd.DataFrame({'Date': dates, 'Miles': miles})
data.set_index('Date', inplace=True)
data.head()

## Visualization

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(data['Miles'])
plt.title('Miles Travelled Over Time')
plt.show()

## Preprocessing

In [ ]:
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(data[['Miles']])

In [ ]:
def create_sequences(data, seq_length=30):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i])
        y.append(data[i])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data)

In [ ]:
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(X_train.shape, X_test.shape)

## Build LSTM Model

In [ ]:
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model.add(LSTM(50))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

## Train Model

In [ ]:
model.fit(X_train, y_train, epochs=15, batch_size=32)

## Predictions

In [ ]:
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)
y_test_actual = scaler.inverse_transform(y_test)

## Visualization of Results

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(y_test_actual, label='Actual Miles')
plt.plot(predictions, label='Predicted Miles')
plt.legend()
plt.title('Miles Travelled Prediction')
plt.show()

## Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(y_test_actual, predictions))
print('RMSE:', rmse)